In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/Chemit797/PepADMET-Dataset/main/PEPlife/PEPlife_half_life_all.csv"

df = pd.read_csv(url)

df_clean = df[["SEQUENCE", "Converted Half-life(seconds)"]].copy()
df_clean.columns = ["sequence", "half_life"]

df_clean = df_clean.dropna()

df_clean["sequence"] = (
    df_clean["sequence"]
    .astype(str)
    .str.upper()
    .str.strip()
)

standard_aa = set("ACDEFGHIKLMNPQRSTVWY")

def is_valid_peptide(seq):
    return all(aa in standard_aa for aa in seq)

df_clean = df_clean[df_clean["sequence"].apply(is_valid_peptide)]

df_clean = df_clean[
    df_clean["sequence"].apply(lambda x: 5 <= len(x) <= 50)
]

df_clean = df_clean.drop_duplicates(subset="sequence")
df_clean = df_clean.reset_index(drop=True)

print("Clean dataset shape:", df_clean.shape)
df_clean.head()

Clean dataset shape: (722, 2)


,sequence,half_life
0,AAGIGILTV,22.0000
1,GSIGAASMEF,2.7013
2,IGAASMEFCF,0.0629
3,AASMEFCFDV,0.0556
4,SMEFCFDVFK,0.0103


In [3]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

model_name = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_name)
esm_model = AutoModel.from_pretrained(model_name)

esm_model.eval()

def get_embedding(sequence):
    inputs = tokenizer(
        sequence,
        return_tensors="pt",
        padding=False,
        truncation=False
    )

    with torch.no_grad():
        outputs = esm_model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embedding

D:\anaconda\envs\peptide_ml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\anaconda\envs\peptide_ml\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
W0521 08:15:05.650000 16780 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
from tqdm import tqdm

embeddings = []

for seq in tqdm(df_clean["sequence"]):
    emb = get_embedding(seq)
    embeddings.append(emb)

X = np.array(embeddings)
y = df_clean["half_life"].values

print("X shape:", X.shape)
print("y shape:", y.shape)

100%|██████████| 722/722 [00:07<00:00, 99.50it/s] 

X shape: (722, 320)
y shape: (722,)


In [5]:
from sklearn.model_selection import (
    train_test_split,
    RepeatedKFold,
    RandomizedSearchCV
)

from sklearn.ensemble import ExtraTreesRegressor

from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)

y_log = np.log10(y + 1)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_log,
    test_size=0.2,
    random_state=42
)

base_model = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1
)

param_dist = {

    "n_estimators": [300, 500, 800],

    "max_depth": [10, 20, 30],

    "min_samples_split": [2, 5, 10],

    "min_samples_leaf": [1, 2, 4],

    "max_features": [
        "sqrt",
        0.3,
        0.5,
        0.7
    ]
}

cv = RepeatedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring="r2",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=0
)

search.fit(X_train, y_train)

best_model = search.best_estimator_

print("Best parameters:")
print(search.best_params_)

print("\nBest CV R2:")
print(search.best_score_)

pred = best_model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    pred
)

r2 = r2_score(
    y_test,
    pred
)

print("\nTest results:")
print("MAE:", mae)
print("R2:", r2)

Best parameters:
{'n_estimators': 800, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20}

Best CV R2:
0.7308446761606621

Test results:
MAE: 0.5357614544143905
R2: 0.7522527193322555


In [10]:
import joblib

joblib.dump(
    best_model,
    "half_life_model.pkl"
)

['half_life_model.pkl']

In [14]:
from sklearn.model_selection import cross_validate, RepeatedKFold

cv = RepeatedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

scores = cross_validate(
    best_model,
    X,
    y_log,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

print("R2 scores:")
print(scores["test_score"])

print(
    "\nMean:",
    scores["test_score"].mean()
)

print(
    "Std:",
    scores["test_score"].std()
)

R2 scores:
[0.75144951 0.79939861 0.69674366 0.69453235 0.73713686 0.7732102
 0.79642057 0.74650071 0.7239166  0.73204498 0.77621982 0.70523884
 0.75423298 0.72070765 0.77531633]

Mean: 0.7455379778485586
Std: 0.032800923820524845


In [16]:
joblib.dump(best_model, "hl_model.pkl")

np.save("X_esm2.npy", X)
np.save("y_hl.npy", y)

df_clean.to_csv("hl_clean_data.csv", index=False)

In [24]:
bpc157_sequence="GEPPPGKPADDAGLV"

amino_acids=list(
    "ACDEFGHIKLMNPQRSTVWY"
)

mutants=[]

for pos, old_aa in enumerate(
        bpc157_sequence,
        start=1
):

    for new_aa in amino_acids:

        if new_aa==old_aa:
            continue

        mutant_sequence=(

            bpc157_sequence[:pos-1]
            + new_aa +
            bpc157_sequence[pos:]

        )

        mutants.append({

            "id":
            f"BPC157_{old_aa}{pos}{new_aa}",

            "position":
            pos,

            "original_aa":
            old_aa,

            "mutated_aa":
            new_aa,

            "mutant_sequence":
            mutant_sequence
        })

df_mutants=pd.DataFrame(
    mutants
)

print(
    "Number of mutants:",
    len(df_mutants)
)

Number of mutants: 285


In [138]:
mutant_embeddings = []

for seq in tqdm(df_mutants["mutant_sequence"]):
    emb = get_embedding(seq)
    mutant_embeddings.append(emb)

X_mutants = np.array(mutant_embeddings)

pred_log = best_model.predict(X_mutants)

pred_half_life_seconds = (10 ** pred_log) - 1

original_emb = get_embedding(bpc157_sequence)

original_emb = np.array([original_emb])

original_pred_log = best_model.predict(
    original_emb
)

original_half_life = (
    10 ** original_pred_log
) - 1

original_half_life = original_half_life[0]

print(
    "Original BPC157 half-life:",
    round(original_half_life, 2),
    "seconds"
)

df_ranked = df_mutants.copy()
df_ranked["pred_half_life_seconds"] = pred_half_life_seconds
df_ranked["pred_half_life_minutes"] = df_ranked["pred_half_life_seconds"] / 60

df_ranked["delta_seconds"] = (
    df_ranked["pred_half_life_seconds"]
    - original_half_life
)

df_ranked["fold_change"] = (
    df_ranked["pred_half_life_seconds"]
    / original_half_life
)

df_ranked = df_ranked.sort_values(
    "pred_half_life_seconds",
    ascending=False
).reset_index(drop=True)

df_ranked.to_csv("bpc157_mutants_hl.csv", index=False)

df_ranked.head(10)

100%|██████████| 285/285 [00:02<00:00, 113.47it/s]


Original BPC157 half-life: 625.69 seconds


,id,position,original_aa,mutated_aa,mutant_sequence,pred_half_life_seconds,pred_half_life_minutes,delta_seconds,fold_change
0,BPC157_K7E,7,K,E,GEPPPGEPADDAGLV,1540.325450,25.672091,914.631155,2.461786
1,BPC157_G6P,6,G,P,GEPPPPKPADDAGLV,1474.969226,24.582820,849.274931,2.357332
2,BPC157_K7P,7,K,P,GEPPPGPPADDAGLV,1393.846357,23.230773,768.152061,2.227680
3,BPC157_V15P,15,V,P,GEPPPGKPADDAGLP,1293.921071,21.565351,668.226775,2.067976
4,BPC157_V15W,15,V,W,GEPPPGKPADDAGLW,1180.943024,19.682384,555.248729,1.887412
5,BPC157_G1P,1,G,P,PEPPPGKPADDAGLV,1162.635338,19.377256,536.941043,1.858152
6,BPC157_A12P,12,A,P,GEPPPGKPADDPGLV,1161.781661,19.363028,536.087366,1.856788
7,BPC157_L14P,14,L,P,GEPPPGKPADDAGPV,1087.730071,18.128835,462.035776,1.738437
8,BPC157_V15D,15,V,D,GEPPPGKPADDAGLD,1078.384330,17.973072,452.690034,1.723500
9,BPC157_A9P,9,A,P,GEPPPGKPPDDAGLV,1061.690683,17.694845,435.996388,1.696820


In [140]:
df_candidates = (
    df_ranked
    .sort_values(
        "pred_half_life_seconds",
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

print(
    "Selected top candidates:",
    len(df_candidates)
)

print(
    df_candidates[
        [
            "id",
            "pred_half_life_seconds",
            "fold_change"
        ]
    ]
)

Selected top candidates: 20
             id  pred_half_life_seconds  fold_change
0    BPC157_K7E             1540.325450     2.461786
1    BPC157_G6P             1474.969226     2.357332
2    BPC157_K7P             1393.846357     2.227680
3   BPC157_V15P             1293.921071     2.067976
4   BPC157_V15W             1180.943024     1.887412
5    BPC157_G1P             1162.635338     1.858152
6   BPC157_A12P             1161.781661     1.856788
7   BPC157_L14P             1087.730071     1.738437
8   BPC157_V15D             1078.384330     1.723500
9    BPC157_A9P             1061.690683     1.696820
10  BPC157_V15C             1056.291236     1.688191
11   BPC157_K7D             1032.238441     1.649749
12  BPC157_L14D             1003.823260     1.604335
13   BPC157_G1D              998.167249     1.595295
14   BPC157_K7A              992.411213     1.586096
15   BPC157_G6E              979.114911     1.564846
16   BPC157_K7Q              963.359683     1.539665
17  BPC157_A12G   

In [142]:
with open("bpc157_mutants.fasta", "w") as f:
    for _, row in df_ranked.iterrows():
        f.write(f">{row['id']}\n")
        f.write(f"{row['mutant_sequence']}\n")

In [144]:
!toxinpred3 -i bpc157_mutants.fasta -o toxinpred3_results.csv -m 1 -d 2

"toxinpred3" не является внутренней или внешней
командой, исполняемой программой или пакетным файлом.


In [146]:
tox = pd.read_csv(
    "toxinpred3_results.csv"
)

tox_small = tox[
    [
        "ID",
        "ML Score",
        "Prediction",
        "PPV"
    ]
].copy()

tox_small.columns = [

    "id",

    "toxicity_score",

    "toxicity_prediction",

    "toxicity_ppv"
]


df_ranked_tox = df_ranked.merge(
    tox_small,
    on="id",
    how="left"
)


print(
    "Missing toxicity values:",
    df_ranked_tox[
        "toxicity_score"
    ].isna().sum()
)

Missing toxicity values: 0


In [176]:
df_ranked_tox = (
    df_ranked_tox
    .sort_values(
        "pred_half_life_seconds",
        ascending=False
    )
    .reset_index(drop=True)
)


df_ranked_tox.to_csv(
    "bpc157_mutants_with_toxicity.csv",
    index=False
)


df_ranked_tox[
[
    "id",

    "mutant_sequence",

    "pred_half_life_seconds",

    "delta_seconds",

    "fold_change",

    "toxicity_score",

    "toxicity_prediction",

    "toxicity_ppv"

]
].head(20)

,id,mutant_sequence,pred_half_life_seconds,delta_seconds,fold_change,toxicity_score,toxicity_prediction,toxicity_ppv
0,BPC157_K7E,GEPPPGEPADDAGLV,1540.325450,914.631155,2.461786,0.130,Non-Toxin,0.042233
1,BPC157_G6P,GEPPPPKPADDAGLV,1474.969226,849.274931,2.357332,0.105,Non-Toxin,0.011380
2,BPC157_K7P,GEPPPGPPADDAGLV,1393.846357,768.152061,2.227680,0.150,Non-Toxin,0.066915
3,BPC157_V15P,GEPPPGKPADDAGLP,1293.921071,668.226775,2.067976,0.200,Non-Toxin,0.128620
4,BPC157_V15W,GEPPPGKPADDAGLW,1180.943024,555.248729,1.887412,0.230,Non-Toxin,0.165643
5,BPC157_G1P,PEPPPGKPADDAGLV,1162.635338,536.941043,1.858152,0.160,Non-Toxin,0.079256
6,BPC157_A12P,GEPPPGKPADDPGLV,1161.781661,536.087366,1.856788,0.180,Non-Toxin,0.103938
7,BPC157_L14P,GEPPPGKPADDAGPV,1087.730071,462.035776,1.738437,0.185,Non-Toxin,0.110108
8,BPC157_V15D,GEPPPGKPADDAGLD,1078.384330,452.690034,1.723500,0.140,Non-Toxin,0.054574
9,BPC157_A9P,GEPPPGKPPDDAGLV,1061.690683,435.996388,1.696820,0.195,Non-Toxin,0.122450


In [178]:
df_candidates = (
    df_ranked_tox
    .sort_values(
        "pred_half_life_seconds",
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

print(
    "Selected candidates:",
    len(df_candidates)
)

print(
    df_candidates[
        [
            "id",
            "mutant_sequence",
            "pred_half_life_seconds",
            "delta_seconds",
            "fold_change",
            "toxicity_score",
            "toxicity_prediction",
            "toxicity_ppv"
        ]
    ]
)

Selected candidates: 20
             id  mutant_sequence  pred_half_life_seconds  delta_seconds  \
0    BPC157_K7E  GEPPPGEPADDAGLV             1540.325450     914.631155   
1    BPC157_G6P  GEPPPPKPADDAGLV             1474.969226     849.274931   
2    BPC157_K7P  GEPPPGPPADDAGLV             1393.846357     768.152061   
3   BPC157_V15P  GEPPPGKPADDAGLP             1293.921071     668.226775   
4   BPC157_V15W  GEPPPGKPADDAGLW             1180.943024     555.248729   
5    BPC157_G1P  PEPPPGKPADDAGLV             1162.635338     536.941043   
6   BPC157_A12P  GEPPPGKPADDPGLV             1161.781661     536.087366   
7   BPC157_L14P  GEPPPGKPADDAGPV             1087.730071     462.035776   
8   BPC157_V15D  GEPPPGKPADDAGLD             1078.384330     452.690034   
9    BPC157_A9P  GEPPPGKPPDDAGLV             1061.690683     435.996388   
10  BPC157_V15C  GEPPPGKPADDAGLC             1056.291236     430.596941   
11   BPC157_K7D  GEPPPGDPADDAGLV             1032.238441     406.544146   
1

In [182]:
df_candidates.to_csv(
    "top20_candidates.csv",
    index=False
)

In [198]:
solubility_results = {
    "BPC157_G6P": -2,
    "BPC157_G6E": -3,
    "BPC157_G1P": -2,
    "BPC157_K7E": -4,
    "BPC157_K7P": -3,
    "BPC157_V15P": -2,
    "BPC157_V15D": -3,
    "BPC157_A12P": -2,
    "BPC157_V15W": -2,
    "BPC157_A9P": -2,
    "BPC157_V15C": -2.1,
    "BPC157_G6D": -3,
    "BPC157_K7D": -4,
    "BPC157_L14P": -2,
    "BPC157_L14D": -3,
    "BPC157_K7A": -3,
    "BPC157_K7Q": -3,
    "BPC157_A12G": -2,
    "BPC157_G1D": -3,
    "BPC157_G13P": -2,    
}

In [214]:
df_candidates["solubility_score"] = (
    df_candidates["id"]
    .map(solubility_results)
)

print(
    "Missing:",
    df_candidates[
        "solubility_score"
    ].isna().sum()
)

df_candidates["solubility_positive"] = (
    -df_candidates[
        "solubility_score"
    ]
)

mask = (
    df_candidates[
        "solubility_positive"
    ].notna()
)

scaler = MinMaxScaler()

df_candidates.loc[
    mask,
    "solubility_norm"
] = scaler.fit_transform(

    df_candidates.loc[
        mask,
        ["solubility_positive"]
    ]
)

df_candidates.drop(
    columns=[
        "solubility_positive"
    ],
    inplace=True
)

print(
    df_candidates[
        [
            "id",
            "pred_half_life_seconds",
            "toxicity_score",
            "solubility_score",
            "solubility_norm"
        ]
    ].head(20)
)

Candidates: 20
             id  pred_half_life_seconds  toxicity_score  \
0    BPC157_K7E             1540.325450           0.130   
1   BPC157_V15P             1293.921071           0.200   
2    BPC157_K7P             1393.846357           0.150   
3   BPC157_V15D             1078.384330           0.140   
4   BPC157_L14P             1087.730071           0.185   
5    BPC157_G1D              998.167249           0.130   
6   BPC157_L14D             1003.823260           0.185   
7    BPC157_K7D             1032.238441           0.110   
8    BPC157_G6P             1474.969226           0.105   
9    BPC157_G6D              959.789191           0.085   
10   BPC157_K7Q              963.359683           0.135   
11  BPC157_A12P             1161.781661           0.180   
12   BPC157_G1P             1162.635338           0.160   
13   BPC157_G6E              979.114911           0.095   
14   BPC157_A9P             1061.690683           0.195   
15  BPC157_V15W             1180.943024  

In [188]:
waltz = pd.read_excel(
    "waltz.xlsx"
)

waltz = waltz[
    ["Sequence","Classification"]
].copy()

waltz.columns = [
    "sequence",
    "label"
]

waltz["label"] = (
    waltz["label"]
    .map({
        "amyloid":1,
        "non-amyloid":0
    })
)

waltz = waltz.dropna()

waltz["sequence"]=(

    waltz["sequence"]
    .astype(str)
    .str.upper()
    .str.strip()

)

standard_aa=set(
    "ACDEFGHIKLMNPQRSTVWY"
)

def is_valid_peptide(seq):

    return all(
        aa in standard_aa
        for aa in seq
    )

waltz=waltz[
    waltz["sequence"]
    .apply(
        is_valid_peptide
    )
]

waltz=(
    waltz
    .drop_duplicates(
        subset="sequence"
    )
    .reset_index(
        drop=True
    )
)

print(
    "Dataset:",
    waltz.shape
)

print(
    waltz[
        "label"
    ].value_counts(
        normalize=True
    )
)

Dataset: (1403, 2)
label
0.0    0.63578
1.0    0.36422
Name: proportion, dtype: float64


In [190]:
from sklearn.model_selection import train_test_split

from sklearn.ensemble import ExtraTreesClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report
)

embeddings=[]

for seq in tqdm(
        waltz["sequence"]
):

    embeddings.append(
        get_embedding(seq)
    )

X=np.array(
    embeddings
)

y=waltz[
    "label"
].values

X_train,\
X_test,\
y_train,\
y_test=(

train_test_split(

X,
y,

test_size=0.2,

random_state=42,

stratify=y
)

)

base_model=(
    ExtraTreesClassifier(

        random_state=42,

        n_jobs=-1,

        class_weight="balanced"
    )
)


param_dist={

    "n_estimators":[
        300,
        500,
        800
    ],

    "max_depth":[
        None,
        10,
        20
    ],

    "min_samples_split":[
        2,
        5,
        10
    ],

    "min_samples_leaf":[
        1,
        2,
        4
    ],

    "max_features":[
        "sqrt",
        0.5
    ]
}


cv=RepeatedKFold(

    n_splits=5,

    n_repeats=3,

    random_state=42
)


search=RandomizedSearchCV(

    estimator=base_model,

    param_distributions=param_dist,

    n_iter=20,

    scoring="roc_auc",

    cv=cv,

    n_jobs=-1,

    random_state=42,

    verbose=0
)

search.fit(
    X_train,
    y_train
)

best_agg_model=(
    search
    .best_estimator_
)

print(
    "Best params:"
)

print(
    search.best_params_
)

print(
    "Best CV ROC-AUC:"
)

print(
    search.best_score_
)

pred=(
    best_agg_model
    .predict(
        X_test
    )
)

pred_prob=(
    best_agg_model
    .predict_proba(
        X_test
    )[:,1]
)

print(
    "\nAccuracy:",
    accuracy_score(
        y_test,
        pred
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        pred_prob
    )
)

print(
    classification_report(
        y_test,
        pred
    )
)

scores=cross_validate(

    best_agg_model,

    X,

    y,

    cv=cv,

    scoring="roc_auc",

    n_jobs=-1
)

print(
    "\nMean ROC-AUC:"
)

print(
    scores[
        "test_score"
    ].mean()
)

print(
    "Std:"
)

print(
    scores[
        "test_score"
    ].std()
)

100%|██████████| 1403/1403 [00:12<00:00, 111.80it/s]


Best params:
{'n_estimators': 800, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': None}
Best CV ROC-AUC:
0.8865799979888134

Accuracy: 0.8042704626334519
ROC-AUC: 0.9108883776974477
              precision    recall  f1-score   support

         0.0       0.80      0.93      0.86       179
         1.0       0.82      0.59      0.69       102

    accuracy                           0.80       281
   macro avg       0.81      0.76      0.77       281
weighted avg       0.81      0.80      0.80       281


Mean ROC-AUC:
0.8899699240294119
Std:
0.01728689163933392


In [192]:
mutant_embeddings = []

for seq in tqdm(

    df_candidates[
        "mutant_sequence"
    ]

):

    mutant_embeddings.append(
        get_embedding(seq)
    )


X_mutants = np.array(
    mutant_embeddings
)


df_candidates[
    "aggregation_probability"
] = (

    best_agg_model
    .predict_proba(
        X_mutants
    )[:,1]

)

scaler = MinMaxScaler()

df_candidates[
    "aggregation_norm"
] = (

    1 -
    scaler.fit_transform(

        df_candidates[
            [
                "aggregation_probability"
            ]
        ]

    )

)


print(

df_candidates[
[
    "id",

    "aggregation_probability",

    "aggregation_norm"
]]

)

100%|██████████| 285/285 [00:02<00:00, 116.98it/s]


              id  aggregation_probability  aggregation_norm
0     BPC157_K7E                  0.23625              0.05
1     BPC157_G6P                  0.22625              0.13
2     BPC157_K7P                  0.20875              0.27
3    BPC157_V15P                  0.14000              0.82
4    BPC157_V15W                  0.17250              0.56
..           ...                      ...               ...
280  BPC157_D11M                  0.13250              0.88
281   BPC157_P8F                  0.14875              0.75
282   BPC157_P4F                  0.15750              0.68
283   BPC157_P3F                  0.14750              0.76
284   BPC157_P8I                  0.17750              0.52

[285 rows x 3 columns]


In [202]:
proteolysis_results = {
    "BPC157_G6P": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 2,
        "elastase": 3
    },

    "BPC157_G6E": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 2,
        "elastase": 3
    },

    "BPC157_G1P": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 2,
        "elastase": 3
    },

    "BPC157_K7E": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 2,
        "elastase": 3
    },

    "BPC157_K7P": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 2,
        "elastase": 3
    },

    "BPC157_V15P": {
        "trypsin": 0,
        "chymotrypsin": 0,
        "pepsin": 1,
        "elastase": 2
    },

    "BPC157_V15D": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 2,
        "elastase": 2
    },

    "BPC157_G6D": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 2,
        "elastase": 3
    },

    "BPC157_A9P": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 2,
        "elastase": 2
    },

    "BPC157_V15W": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 3,
        "elastase": 2
    },

    "BPC157_A12P": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 1,
        "elastase": 2
    },

    "BPC157_V15C": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 2,
        "elastase": 2
    },

    "BPC157_L14P": {
        "trypsin": 0,
        "chymotrypsin": 0,
        "pepsin": 0,
        "elastase": 3
    },

    "BPC157_K7D": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 2,
        "elastase": 3
    },

    "BPC157_G1D": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 2,
        "elastase": 3
    },

    "BPC157_L14D": {
        "trypsin": 0,
        "chymotrypsin": 0,
        "pepsin": 0,
        "elastase": 3
    },

    "BPC157_K7A": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 2,
        "elastase": 4
    },

    "BPC157_K7Q": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 2,
        "elastase": 3
    },

    "BPC157_A12G": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 2,
        "elastase": 2
    },

    "BPC157_G13P": {
        "trypsin": 0,
        "chymotrypsin": 1,
        "pepsin": 1,
        "elastase": 3
    }
}

In [204]:
proteolysis_df = pd.DataFrame.from_dict(
    proteolysis_results,
    orient="index"
).reset_index()

proteolysis_df = proteolysis_df.rename(
    columns={"index": "id"}
)

proteolysis_df.head()

,id,trypsin,chymotrypsin,pepsin,elastase
0,BPC157_G6P,0,1,2,3
1,BPC157_G6E,0,1,2,3
2,BPC157_G1P,0,1,2,3
3,BPC157_K7E,0,1,2,3
4,BPC157_K7P,0,1,2,3


In [206]:
df_candidates = df_candidates.merge(
    proteolysis_df,
    on="id",
    how="left"
)

In [208]:
protease_cols = [
    "trypsin",
    "chymotrypsin",
    "pepsin",
    "elastase"
]

scaler = MinMaxScaler()

norm_cols = [
    c+"_norm"
    for c in protease_cols
]

df_candidates[norm_cols] = (
    scaler.fit_transform(
        df_candidates[
            protease_cols
        ]
    )
)

for c in protease_cols:

    df_candidates[
        c+"_norm"
    ] = (
        1-
        df_candidates[
            c+"_norm"
        ]
    )

df_candidates[
    "proteolysis_score"
] = (

    df_candidates[
        [
            "trypsin_norm",
            "chymotrypsin_norm",
            "pepsin_norm",
            "elastase_norm"
        ]
    ]
    .mean(axis=1)

)

print(

    df_candidates[
        [
            "id",
            "trypsin",
            "chymotrypsin",
            "pepsin",
            "elastase",
            "proteolysis_score"
        ]
    ]
    .head(20)

)

             id  trypsin  chymotrypsin  pepsin  elastase  proteolysis_score
0    BPC157_K7E      0.0           1.0     2.0       3.0           0.458333
1    BPC157_G6P      0.0           1.0     2.0       3.0           0.458333
2    BPC157_K7P      0.0           1.0     2.0       3.0           0.458333
3   BPC157_V15P      0.0           0.0     1.0       2.0           0.916667
4   BPC157_V15W      0.0           1.0     3.0       2.0           0.500000
5    BPC157_G1P      0.0           1.0     2.0       3.0           0.458333
6   BPC157_A12P      0.0           1.0     1.0       2.0           0.666667
7   BPC157_L14P      0.0           0.0     0.0       3.0           0.875000
8   BPC157_V15D      0.0           1.0     2.0       2.0           0.583333
9    BPC157_A9P      0.0           1.0     2.0       2.0           0.583333
10  BPC157_V15C      0.0           1.0     2.0       2.0           0.583333
11   BPC157_K7D      0.0           1.0     2.0       3.0           0.458333
12  BPC157_L

In [218]:
scaler = MinMaxScaler()

df_candidates["half_life_norm"] = (
    scaler.fit_transform(
        df_candidates[
            ["pred_half_life_seconds"]
        ]
    )
)

df_candidates["toxicity_norm"] = (
    1 -
    scaler.fit_transform(
        df_candidates[
            ["toxicity_score"]
        ]
    )
)

df_candidates["aggregation_norm"] = (
    1 -
    scaler.fit_transform(
        df_candidates[
            ["aggregation_probability"]
        ]
    )
)

weights = {

    "half_life_norm":0.40,

    "toxicity_norm":0.15,

    "solubility_norm":0.15,

    "aggregation_norm":0.15,

    "proteolysis_score":0.15
}

df_candidates["final_score"]=(

    df_candidates["half_life_norm"]
    *weights["half_life_norm"]

    +

    df_candidates["toxicity_norm"]
    *weights["toxicity_norm"]

    +

    df_candidates["solubility_norm"]
    *weights["solubility_norm"]

    +

    df_candidates["aggregation_norm"]
    *weights["aggregation_norm"]

    +

    df_candidates["proteolysis_score"]
    *weights["proteolysis_score"]

)

df_final=(

    df_candidates

    .sort_values(
        "final_score",
        ascending=False
    )

    .reset_index(
        drop=True
    )

)

print(

df_final[
[
"id",

"pred_half_life_seconds",

"toxicity_score",

"aggregation_probability",

"solubility_score",

"proteolysis_score",

"final_score"
]]

)

df_final.to_csv(
    "bpc157_final_ranked.csv",
    index=False
)

             id  pred_half_life_seconds  toxicity_score  \
0    BPC157_K7E             1540.325450           0.130   
1   BPC157_V15P             1293.921071           0.200   
2    BPC157_K7P             1393.846357           0.150   
3    BPC157_G6P             1474.969226           0.105   
4   BPC157_L14P             1087.730071           0.185   
5   BPC157_V15D             1078.384330           0.140   
6   BPC157_A12P             1161.781661           0.180   
7    BPC157_G1P             1162.635338           0.160   
8   BPC157_V15W             1180.943024           0.230   
9    BPC157_G1D              998.167249           0.130   
10   BPC157_K7D             1032.238441           0.110   
11  BPC157_L14D             1003.823260           0.185   
12   BPC157_A9P             1061.690683           0.195   
13   BPC157_G6D              959.789191           0.085   
14   BPC157_K7Q              963.359683           0.135   
15   BPC157_G6E              979.114911           0.095 

In [226]:
from sklearn.metrics.pairwise import cosine_similarity

wild_type = "GEPPPGKPADDAGLV"

top5 = (
    df_final
    .head(5)
    .copy()
)

wt_embedding = (
    get_embedding(
        wild_type
    )
    .reshape(1,-1)
)

top_embeddings=[]

for seq in tqdm(
    top5[
        "mutant_sequence"
    ]
):

    top_embeddings.append(
        get_embedding(seq)
    )


top_embeddings=np.array(
    top_embeddings
)

similarities=(
    cosine_similarity(
        wt_embedding,
        top_embeddings
    )[0]
)


top5[
    "cosine_to_wt"
]=similarities


print(

top5[
[
"id",
"mutant_sequence",
"cosine_to_wt"
]]

.sort_values(
    "cosine_to_wt",
    ascending=False
)
)

100%|██████████| 5/5 [00:00<00:00, 116.24it/s]

            id  mutant_sequence  cosine_to_wt
1  BPC157_V15P  GEPPPGKPADDAGLP      0.992365
4  BPC157_L14P  GEPPPGKPADDAGPV      0.989750
3   BPC157_G6P  GEPPPPKPADDAGLV      0.988029
2   BPC157_K7P  GEPPPGPPADDAGLV      0.979712
0   BPC157_K7E  GEPPPGEPADDAGLV      0.971090


In [228]:
sim_matrix = cosine_similarity(
    top_embeddings
)

sim_df = pd.DataFrame(
    sim_matrix,
    index=top5["id"],
    columns=top5["id"]
)

print(
    sim_df.round(3)
)

id           BPC157_K7E  BPC157_V15P  BPC157_K7P  BPC157_G6P  BPC157_L14P
id                                                                       
BPC157_K7E        1.000        0.971       0.994       0.964        0.972
BPC157_V15P       0.971        1.000       0.981       0.985        0.996
BPC157_K7P        0.994        0.981       1.000       0.974        0.980
BPC157_G6P        0.964        0.985       0.974       1.000        0.984
BPC157_L14P       0.972        0.996       0.980       0.984        1.000


In [230]:
import itertools

wild_type = "GEPPPGKPADDAGLV"

top_mutations = {
    "V15P":(15,"P"),
    "L14P":(14,"P"),
    "G6P":(6,"P"),
    "K7P":(7,"P"),
    "K7E":(7,"E")
}

combination_rows=[]

for r in range(
    2,
    len(top_mutations)+1
):
    for combo in itertools.combinations(
        top_mutations.items(),
        r
    ):
        sequence=list(
            wild_type
        )
        mutation_names=[]
        positions=[]
        conflict=False

        for name,(pos,new_aa) in combo:
            idx=pos-1

            if idx in positions:
                conflict=True
                break

            sequence[idx]=new_aa
            positions.append(
                idx
            )
            mutation_names.append(
                name
            )

        if conflict:
            continue

        combination_rows.append(
            {
                "id":
                "BPC157_"+"_".join(
                    mutation_names
                ),
                "n_mutations":
                len(
                    mutation_names
                ),
                "mutant_sequence":
                "".join(
                    sequence
                )
            }
        )

df_combinations=(
    pd.DataFrame(
        combination_rows
    )
)

print(
    "Number:",
    len(df_combinations)
)

print(
    df_combinations
)

Number: 18
                          id  n_mutations  mutant_sequence
0           BPC157_V15P_L14P            2  GEPPPGKPADDAGPP
1            BPC157_V15P_G6P            2  GEPPPPKPADDAGLP
2            BPC157_V15P_K7P            2  GEPPPGPPADDAGLP
3            BPC157_V15P_K7E            2  GEPPPGEPADDAGLP
4            BPC157_L14P_G6P            2  GEPPPPKPADDAGPV
5            BPC157_L14P_K7P            2  GEPPPGPPADDAGPV
6            BPC157_L14P_K7E            2  GEPPPGEPADDAGPV
7             BPC157_G6P_K7P            2  GEPPPPPPADDAGLV
8             BPC157_G6P_K7E            2  GEPPPPEPADDAGLV
9       BPC157_V15P_L14P_G6P            3  GEPPPPKPADDAGPP
10      BPC157_V15P_L14P_K7P            3  GEPPPGPPADDAGPP
11      BPC157_V15P_L14P_K7E            3  GEPPPGEPADDAGPP
12       BPC157_V15P_G6P_K7P            3  GEPPPPPPADDAGLP
13       BPC157_V15P_G6P_K7E            3  GEPPPPEPADDAGLP
14       BPC157_L14P_G6P_K7P            3  GEPPPPPPADDAGPV
15       BPC157_L14P_G6P_K7E            3  GE